In [ ]:
import pandas as pd
import torch
import pydicom
import numpy as np
import os
from pathlib import Path

np.random.seed(42)
torch.manual_seed(42)

In [ ]:
#Data preparation
root_path = '/kaggle/input/rsna-intracranial-aneurysm-detection'
train_df = pd.read_csv(os.path.join(root_path,'train.csv'))
train_loc_df = pd.read_csv(os.path.join(root_path,'train_localizers.csv'))
train_df = pd.read_csv(os.path.join(root_path,'train.csv'))
train_loc_df = pd.read_csv(os.path.join(root_path,'train_localizers.csv'))
series_path = os.path.join(root_path,'series')
classification_df = dict()
index_map = []
for i, row in train_df.iterrows():
    series_id = row.iloc[0]
    aneurysm = row.iloc[-1]
    for dcm_id in os.listdir(os.path.join(series_path, series_id)):
        dcm_id = Path(dcm_id).stem
        classification_df[(series_id, dcm_id)] = 0
        index_map.append(series_id, dcm_id)
    if aneurysm == 1:
        for loc_id, loc_row in train_loc_df[train_loc_df.iloc[:,0]==series_id].iterrows():
            classification_df[(series_id, loc_row.iloc[1])] = 1
assert len(classification_df.keys()) == len(index_map)
nb_positive = sum(classification_df.values())

# validation set with balanced classes
positive_indices = [i for i, (series_id, dcm_id) in enumerate(index_map.values()) if classification_df[(series_id, dcm_id)] == 1]
negative_indices = [i for i, (series_id, dcm_id) in enumerate(index_map.values()) if classification_df[(series_id, dcm_id)] == 0]
np.random.shuffle(positive_indices)
np.random.shuffle(negative_indices)
val_positive_indices = positive_indices[:0.2 * len(positive_indices)]
val_negative_indices = negative_indices[:0.2 * len(positive_indices)]
val_indices = val_positive_indices + val_negative_indices
train_indices = [i for i in range(len(index_map)) if i not in val_indices]

In [ ]:
class AneurysmDataset(torch.utils.data.Dataset):
    def __init__(self, root_path, train=True):
        super().__init__()
        self.root_path = root_path
        self.classification_df = classification_df
        self.train = train
        if train:
            self.index_map = {i: index_map[i] for i in train_indices}
        else:
            self.index_map = {i: index_map[i] for i in val_indices}

    def __getitem__(self, index):
        img_id = self.index_map[index] 
        img_path = os.path.join(self.root_path, 'series', img_id[0], img_id[1]+'.dcm')
        target = torch.tensor(self.classification_df[img_id], dtype=torch.float)
        img = torch.tensor(pydicom.dcmread(img_path).pixel_array, dtype=torch.float).unsqueeze(-3)
        return img, target

    def __len__(self):
        return len(self.index_map.keys())
    
def train_loader(root_path, alpha=2, batch_size=32):
    """ Create a DataLoader for the training set with balanced classes.
    Args:
        root_path (str): Path to the dataset root directory.
        alpha (float): Weighting factor for the negative class.
        batch_size (int): Size of the batches to be returned by the DataLoader.
    Returns:
        torch.utils.data.DataLoader: DataLoader for the training set with balanced classes.
    """
    dataset = AneurysmDataset(root_path, train=True)
    # sampler to ensure balanced classes in each batch
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=[1.0 / len(positive_indices) if dataset.classification_df[dataset.index_map[i]] == 1 else alpha / len(negative_indices) for i in range(len(dataset))],
        num_samples=len(dataset),
        replacement=True
    )
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler)

In [67]:
dataset = AneurysmDataset('/kaggle/input/rsna-intracranial-aneurysm-detection')

In [74]:
print(dataset[1][0].size())
len(dataset)

In [70]:
dl = torch.utils.data.DataLoader(dataset,12)
for i, l in dl:
    print(i, l)
    break

In [ ]:
#Train loop

def train_one_epoch(model, dataloader, optimizer, loss_fn):
    model.train()
    for images, labels in dataloader:
        preds = model(images)
        loss = loss_fn(preds,labels.unsqueeze(1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

def test_model(model, dataloader, loss_fn):
    model.eval()
    all_preds = torch.empty(0)
    all_labels = torch.empty(0)
    with torch.no_grad():
        for images, labels in dataloader:
            all_preds = torch.cat(all_preds, model(images))
            all_labels = torch.cat(all_labels, labels.unsqueeze(1))
        loss = loss_fn(all_preds, all_labels)
    return loss
    
def train(model, dataset, optimizer, loss_fn, epochs):
    val_loader = torch.utils.data.Dataloader(AneurysmDataset('/kaggle/input/rsna-intracranial-aneurysm-detection', train = False), 16)
    train_loader = train_loader('/kaggle/input/rsna-intracranial-aneurysm-detection', alpha=2, batch_size=32)
    
    for epoch in range(epochs):
        train_one_epoch(model, train_loader, optimizer, loss_fn)
        loss = test_model(model, val_loader, loss_fn)
        print(loss)
    

In [85]:
# Model

class AneurysmClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(1, 8, 3)
        self.conv2 = torch.nn.Conv2d(8, 64, 3)
        self.pool = torch.nn.MaxPool2d(2)
        self.fc = torch.nn.Linear(64,1)

    def forward(self, x):
        x = torch.nn.functional.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.nn.functional.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.mean(x,[2,3])
        x = torch.nn.functional.dropout(x, 0.3)
        x = self.fc(x)
        return x
        

In [ ]:
#TEST
model = AneurysmClassifier()
pos_weights = torch.tensor((len(negative_indices) / len(positive_indices)) * 0.5, dtype=torch.float)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weights)
optimizer = torch.optim.AdamW(model.parameters())
dataset = AneurysmDataset('/kaggle/input/rsna-intracranial-aneurysm-detection')

In [90]:
train(model, dataset, optimizer, loss_fn, epochs = 1)

In [7]:
import pydicom
import matplotlib.pyplot as plt

dcm = '/kaggle/input/rsna-intracranial-aneurysm-detection/series/1.2.826.0.1.3680043.8.498.10046318991957083423208748012349179640/1.2.826.0.1.3680043.8.498.11550049804049536118256417115804980060.dcm'

ds = pydicom.dcmread(dcm)
print(ds)
plt.imshow(ds.pixel_array)
plt.show()